Берем все из предыдущего урока

In [43]:
import pandas as pd
# Читаем файл melb_data.csv
melb_data = pd.read_csv('data/melb_data.csv', sep=',')

In [44]:
melb_df = melb_data.copy()
melb_df['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)
years_sold = melb_df['Date'].dt.year
melb_df['MonthSale'] = melb_df['Date'].dt.month
delta_days = melb_df['Date'] - pd.to_datetime('2016-01-01')
melb_df['AgeBuilding'] = melb_df['Date'].dt.year - melb_df['YearBuilt']
melb_df['AgeBuilding'].astype('int16')
melb_df = melb_df.drop('YearBuilt', axis=1)
melb_df['WeekdaySale'] = melb_df['Date'].dt.dayofweek
weekend_count = melb_df[(melb_df['WeekdaySale'] == 5) | (melb_df['WeekdaySale'] == 6)].shape[0]

В некоторых случаях нам необходимо совершать более сложные манипуляции над столбцами. Например, из столбцов, содержащих в себе некоторый текст, необходимо специальным образом извлечь определенные слова, даты и числа.

Для таких случаев Pandas не имеет специальных методов, однако позволяет расширить свой функционал с помощью пользовательских функций.

Мы можем написать функцию, которая принимает на вход один элемент столбца, каким-то образом его обрабатывает и возвращает результат, после чего применить эту функцию к каждому элементу в столбце с помощью специального метода apply(). В результате будет возвращена серия, элементы которой будут представлять результат работы этой функции.

В наших данных есть столбец с адресами. Проблема в том, что в этом столбце слишком много уникальных значений: почти на каждый объект недвижимости приходится свой уникальный адрес.

In [45]:
print(melb_df['Address'].nunique())

13378


При прогнозировании цены такое большое количество возможных категорий может плохо сказаться на модели, которую мы хотели бы построить на этих данных. Говорят, что такой признак, скорее всего, не имеет статистической значимости, потому что не позволяет разделить данные на группы, которые можно сравнить по целевому признаку.

Обычно такие признаки удаляют, но можно поступить умнее: извлечь из признака характеристику подтипа улицы (улица, шоссе, авеню, бульвар). Для этого сначала посмотрим на структуру адреса.

In [46]:
print(melb_df['Address'].loc[177])
print(melb_df['Address'].loc[1812])
print(melb_df['Address'].loc[9001])

2/119 Railway St N
9/400 Dandenong Rd
172 Danks St


Сначала указывается номер дома и корпус, после указывается название улицы, а в конце — подтип улицы, но в некоторых случаях к подтипу добавляется географическая отметка (N — север, S — юг, и тд), она нам не нужна. Для выделения подтипа улицы можно написать функцию:

In [47]:
# На вход функции поступает строка с адресом
def get_street_type(adress):
    stype_dict = {
        'Avenue': 'Ave',
        'Boulevard': 'Bvd',
        'Parade': 'Pde'
    }
# Создаем список географических отметок
    exclude_list = ['N', 'S', 'W', 'E']
# Метод split() разбивает строку по пробелу
# В результате получим список слов в строке и заносим его в переменную adress_list
    adress_list = adress.split(' ')
# Обрезаем список, оставляя в нем только последний элемент,
# потенциальный подтип улицы, и заносим в переменную street_type
    street_type = adress_list[-1]
# Проверяем, что полученный подтип является географической пометкой.
# Для этого проверяем его на наличие в списке exclude_list
    if street_type in exclude_list:
# Если street_type в exclude_list (является географической пометкой),
# переопределяем ее на второй элемент с конца списка adress_list
        street_type = adress_list[-2]
    street_type = stype_dict.get(street_type, street_type)
# Возвращаем street_type, в котором хранится подтип улицы
    return street_type

Теперь применим эту функцию к столбцу с адресом. Для этого передадим функцию get_street_type в аргумент метода столбца apply(). В результате получим серию, которую положим в переменную street_types:

In [48]:
street_types = melb_df['Address'].apply(get_street_type)
display(street_types)

0        St
1        St
2        St
3        La
4        St
         ..
13575    Cr
13576    Dr
13577    St
13578    St
13579    St
Name: Address, Length: 13580, dtype: object

Функция пишется для одного элемента столбца, а метод apply() применяется к каждому его элементу. Используемая функция ОБЯЗАТЕЛЬНО ДОЛЖНА ИМЕТЬ ВОЗВРАЩАЕМОЕ ЗНАЧЕНИЕ.

In [49]:
print(street_types.nunique())

53


У нас есть 56 уникальных значений, но результат можно улучшить. Сначала посмотрим на частоту каждого подтипа улицы с помощью метода value_counts:

In [50]:
display(street_types.value_counts())

Address
St           8012
Rd           2825
Ct            612
Dr            447
Av            321
Gr            311
Pde           226
Pl            169
Cr            152
Cl            100
La             67
Bvd            66
Tce            47
Ave            41
Wy             40
Cct            25
Hwy            24
Sq             11
Crescent        9
Cir             7
Strand          7
Esplanade       6
Grove           5
Gdns            4
Grn             4
Fairway         4
Mews            4
Crossway        3
Righi           3
Victoria        2
Ridge           2
Crofts          2
Esp             2
Glade           1
Gra             1
Woodland        1
Outlook         1
Hts             1
Highway         1
Athol           1
Summit          1
Grand           1
Res             1
Nook            1
Eyrie           1
Dell            1
East            1
Loop            1
Grange          1
Terrace         1
Cove            1
Qy              1
Corso           1
Name: count, dtype: int64

Из данного вывода можно увидеть, что есть группа наиболее популярных подтипов улиц, а дальше частота подтипов быстро падает.

В таком случае можно применить очень распространенный метод уменьшения количества уникальных категорий — выделить n подтипов, которые встречаются чаще всего, а остальные обозначить как other (другие).

nlargest() — возвращает n наибольших значений из серии. 

Мы хотим отобрать 10 популярных подтипов, поэтому n=10.
Названия извлекаются с помощью атрибута index.

In [51]:
popular_stypes = street_types.value_counts().nlargest(n=10).index
print(popular_stypes)

Index(['St', 'Rd', 'Ct', 'Dr', 'Av', 'Gr', 'Pde', 'Pl', 'Cr', 'Cl'], dtype='object', name='Address')


Теперь введем лямбда-функцию, которая будет проверять, есть ли строка х в этом перечне, и если есть, то функция будет возвращать х, а если нет, то будет возвращать 'other'.

После применим эту функцию к серии street_types, а результат определим в новый столбец.

In [52]:
melb_df['StreetType'] = street_types.apply(lambda x: x if x in popular_stypes else 'other')
display(melb_df['StreetType'])

0           St
1           St
2           St
3        other
4           St
         ...  
13575       Cr
13576       Dr
13577       St
13578       St
13579       St
Name: StreetType, Length: 13580, dtype: object

In [53]:
# Проверим результирующее число уникальных подтипов
print(melb_df['StreetType'].nunique())

11


Теперь нет потребности хранить список Adress, потому что если конкретное местоположение объекта все же и влияет на его стоимость, то оно определяется столбцами Longitude (долгота) и Lattitude (широта).

In [54]:
melb_df = melb_df.drop('Address', axis=1)

Некоторые подтипы выше именуются различным образом, но при этом обозначают одинаковые вещи. Например, Av = Avenue, Bvd = Boulevard, Pde = Parade. В реальных задачах стоит обращать пристальное внимание на результаты преобразований и исправлять неточности в них.

Такие ошибки в данных (обозначение идентичных категорий различными именами) являются одним из видов «грязных данных».

Иногда такие неточности бывает очень сложно отследить, а при наличии большого количества категорий (100+) — практически невозможно.

Задание — очистить данные от грязи, выполнив преобразование. Для этого в функции get_street_type объявляем словарь, ключами которого будут являться полные названия подтипов улиц, а значениями — их сокращения. 
Затем мы в переменной street_type выполняем замену по ключу, если такой ключ существует в нашей переменной street_type (меняем на значение, которое доступно по этому ключу). Если такого ключа не существует, то возвращается исходное значение. 

Если street_type есть среди ключей словаря — возьми сокращенное значение; если нет — оставь как есть.

In [ ]:
def get_street_type(adress):
    stype_dict = {
        'Avenue': 'Ave',
        'Boulevard': 'Bvd',
        'Parade': 'Pde'
    }

    exclude_list = ['N', 'S', 'W', 'E']

    adress_list = adress.split(' ')
    street_type = adress_list[-1]

    if street_type in exclude_list:
        street_type = adress_list[-2]

    street_type = stype_dict.get(street_type, street_type)

    return street_type

РЕКОМЕНДАЦИИ по уменьшению числа уникальных значений в признаке, который описывается категориями:

- Определить (хотя бы на глаз) соотношение числа уникальных категорий интересующего признака к общему числу объектов в таблице. Если это соотношение превышает 30%, то это уже повод задуматься над уменьшением числа категорий;

- Если признак уникален для каждого объекта (например, адрес, имя или название), то такой признак, скорее всего, не имеет статистической значимости. От таких признаков чаще всего избавляются. Однако можно попробовать выделить из этого признака какие-то общие черты (например, как было сделано с подтипами улиц). Например, из названия компании ТОО «Три слепые мыши» можно извлечь ТОО — товарищество с ограниченной ответственностью;

- Если даже после преобразования число уникальных категорий все еще велико, можно попробовать с помощью метода value_counts() оценить, если в данных категории, которые употребляются гораздо реже остальных;

- Если в данных есть категории, которые употребляются реже остальных, можно подобрать число популярных категорий таким образом, чтобы эти категории покрывали большую часть данных;

- По итогу можно совершить преобразование, обозначив категории, не попавшие в число популярных, как другие (other).

In [ ]:
# Создаем функцию, которая принимает в качестве аргумента элемент столбца WeekdaySale,
# возвращает 1, если день выходной, и 0 — если не выходной
def get_weekend(weekday):
    if weekday == 5 or weekday == 6:
        return 1
    return 0
# Создаем новый столбец Weekend, применив функцию к столбцу WeekdaySale
melb_df['Weekend'] = melb_df['WeekdaySale'].apply(get_weekend)
# Вычисляем среднюю цену объекта недвижимости, проданного в выходные дни
wknd_mean = melb_df[melb_df['Weekend'] == 1]['Price'].mean()
display(round(wknd_mean))

1081199

In [ ]:
# Выделяем 49 самых популярных селлеров
popular_sellers = melb_df['SellerG'].value_counts().nlargest(n=49).index
# Преобразуем столбец SellerG, меняя всех, кто не входит в число 49 популярных селлеров на other
melb_df['SellerG'] = melb_df['SellerG'].apply(lambda popular: popular if popular in popular_sellers else 'other')
# Вычисляем, во сколько раз минимальная цена селлера Nelson превосходит минимальную цену всех, кто отмечен как other
display(round((melb_df[melb_df['SellerG']=='Nelson']['Price'].min())/(melb_df[melb_df['SellerG']=='other']['Price'].min()), 2))

1.3